# Phase 2 — 16-Step Expansion Sweep

The Tier-1 extension showed a **−0.424 FID improvement at 16 steps** (6.212 → 5.788) with max_prob + τ=0.5 + cap=0.2. This notebook confirms that result is robust across other (τ, cap, metric) combinations and finds the best 16-step config for FID-50K validation.

## Plan

| Block | Description | Configs |
|-------|-------------|---------|
| **A** | max_prob grid at 16 steps: τ ∈ {0.3, 0.5, 0.7} × cap ∈ {0.1, 0.2, 0.3, 0.5} | 12 (1 already done) |
| **B** | other metrics at 16 steps, τ=0.5, cap ∈ {0.1, 0.2, 0.5}: entropy + margin | 6 |
| **C** (optional) | ultra-aggressive: vanilla + max_prob rejection at steps ∈ {8, 12} | 4 |

**Total: 17 configs (21 with block C).**

**Runtime:** ~3 hours without block C, ~4 hours with it.

Results merge with the existing combined CSV for unified analysis.

## 1. Setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Runtime > Change runtime type > A100 GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ARPG = "/content/drive/MyDrive/ARPG-assets"

import os
COMBINED_CSV = f"{DRIVE_ARPG}/results/pilot-20260421/tier1-extension/combined-results.csv"
assert os.path.exists(COMBINED_CSV), f"Missing combined CSV at {COMBINED_CSV}"
print(f"Found combined results: {COMBINED_CSV}")

In [ ]:
os.chdir('/content')
!rm -rf /content/ARPG
!git clone https://github.com/rshahbazov23/comp447-arpg-private.git /content/ARPG
%cd /content/ARPG

# Symlink assets from Drive
!ln -sfn {DRIVE_ARPG}/weights weights
!ln -sfn {DRIVE_ARPG}/eval eval
!ln -sfn {DRIVE_ARPG}/external external

!pip install -q transformers einops Pillow tqdm numpy scipy tensorflow pandas

In [ ]:
# Sanity check
required = [
    "weights/arpg_300m.pt",
    "weights/vq_ds16_c2i.pt",
    "eval/VIRTUAL_imagenet256_labeled.npz",
    "external/guided-diffusion/evaluations/evaluator.py",
    "scripts/run_16step_sweep.sh",
    "scripts/eval_pilot_sweep.py",
]
for p in required:
    status = "OK" if os.path.exists(p) else "MISSING"
    print(f"  [{status}] {p}")

## 2. Run the sweep (sampling)

Set `RUN_AGGRESSIVE_STEPS=1` to also test 8-step and 12-step regimes.

Safe to interrupt — re-running skips configs that already have an NPZ.

In [ ]:
SWEEP_DIR = "/content/sweep-16step"
!mkdir -p {SWEEP_DIR}

env = {
    'SWEEP_DIR':             SWEEP_DIR,
    'NUM_SAMPLES':           '10000',
    'PER_PROC_BATCH_SIZE':   '32',
    'CFG_SCALE':             '5.0',
    'GLOBAL_SEED':           '0',
    'RUN_AGGRESSIVE_STEPS':  '1',   # set to '0' to skip block C
}
env_prefix = ' '.join(f'{k}={v}' for k, v in env.items())

!{env_prefix} bash scripts/run_16step_sweep.sh

In [ ]:
# Verify NPZ counts
import glob
vanilla_npzs = sorted(glob.glob(f"{SWEEP_DIR}/vanilla/*.npz"))
rejection_npzs = sorted(glob.glob(f"{SWEEP_DIR}/rejection/*.npz"))
print(f"Vanilla NPZs:   {len(vanilla_npzs)}")
print(f"Rejection NPZs: {len(rejection_npzs)}")
!du -sh {SWEEP_DIR}

## 3. FID evaluation

**~90 minutes** on A100.

In [ ]:
SWEEP_CSV = f"{SWEEP_DIR}/16step-results.csv"

!python scripts/eval_pilot_sweep.py \
    --pilot-dir {SWEEP_DIR} \
    --reference-npz eval/VIRTUAL_imagenet256_labeled.npz \
    --guided-diffusion external/guided-diffusion/evaluations/evaluator.py \
    --out-csv {SWEEP_CSV}

## 4. Merge with previous results

In [ ]:
import pandas as pd
import re

def extract_steps(npz_name):
    m = re.search(r'step-(\d+)', npz_name)
    return int(m.group(1)) if m else None

prev = pd.read_csv(COMBINED_CSV)
if 'steps' not in prev.columns:
    prev['steps'] = prev['npz'].apply(extract_steps)
if 'source' not in prev.columns:
    prev['source'] = 'prev'

new = pd.read_csv(SWEEP_CSV)
new['steps'] = new['npz'].apply(extract_steps)
new['source'] = 'phase2'

all_results = pd.concat([prev, new], ignore_index=True)
all_results = all_results[all_results['fid'].notna()].copy()
# De-duplicate by npz filename
all_results = all_results.drop_duplicates(subset=['npz'], keep='last').reset_index(drop=True)

print(f"Previous rows:    {len(prev)}")
print(f"Phase-2 new rows: {len(new)}")
print(f"Combined unique:  {len(all_results)}")

## 5. Analysis — does the 16-step effect hold up?

In [ ]:
# Locate vanilla baselines at each step count
vanilla = all_results[all_results['mode'] == 'vanilla'].copy()
vanilla_by_steps = {int(row['steps']): float(row['fid']) for _, row in vanilla.iterrows()
                    if pd.notna(row['steps'])}
print('Vanilla FIDs by step count:')
for s, f in sorted(vanilla_by_steps.items()):
    print(f'  steps={s}: FID={f:.3f}')

### 5.1 Block A — max_prob at 16 steps, full (τ × cap) grid

In [ ]:
block_a = all_results[
    (all_results['mode'] == 'rejection') &
    (all_results['metric'] == 'max_prob') &
    (all_results['steps'] == 16)
].copy().sort_values(['tau', 'cap'])

# Pivot into a τ × cap table of FIDs
pivot = block_a.pivot_table(values='fid', index='tau', columns='cap', aggfunc='first')
print("Block A: FID-10K for max_prob at 16 steps\n")
print(pivot.round(3))

vanilla_16 = vanilla_by_steps.get(16)
if vanilla_16 is not None:
    print(f"\nVanilla (16 steps): FID={vanilla_16:.3f}")
    print("\nDelta vs vanilla:")
    print((pivot - vanilla_16).round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for tau in sorted(block_a['tau'].dropna().unique()):
    sub = block_a[block_a['tau'] == tau].sort_values('cap')
    ax.plot(sub['cap'], sub['fid'], marker='o', linewidth=2, markersize=8,
            label=f'τ={tau}')
if vanilla_16 is not None:
    ax.axhline(vanilla_16, color='gray', linestyle='--',
               label=f'vanilla (16 steps) = {vanilla_16:.3f}')
ax.set_xlabel('max_reject_rate (cap)')
ax.set_ylabel('FID-10K (lower is better)')
ax.set_title('Block A — max_prob at 16 steps, (τ, cap) grid')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 Block B — entropy and margin at 16 steps (τ=0.5)

In [ ]:
block_b = all_results[
    (all_results['mode'] == 'rejection') &
    (all_results['steps'] == 16) &
    (all_results['tau'] == 0.5)
].copy().sort_values(['metric', 'cap'])

print("Block B: metric comparison at 16 steps, τ=0.5\n")
print(f"{'metric':>10} {'cap':>6} {'FID':>8} {'Δ vs vanilla':>14}")
print('-' * 42)
for _, r in block_b.iterrows():
    delta = r['fid'] - vanilla_16 if vanilla_16 is not None else None
    delta_s = f"{delta:+.3f}" if delta is not None else "n/a"
    marker = "  <--" if delta is not None and delta < 0 else ""
    print(f"{r['metric']:>10} {r['cap']:>6.2f} {r['fid']:>8.3f} {delta_s:>14}{marker}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for metric in sorted(block_b['metric'].dropna().unique()):
    sub = block_b[block_b['metric'] == metric].sort_values('cap')
    ax.plot(sub['cap'], sub['fid'], marker='o', linewidth=2, markersize=8, label=metric)
if vanilla_16 is not None:
    ax.axhline(vanilla_16, color='gray', linestyle='--',
               label=f'vanilla (16 steps) = {vanilla_16:.3f}')
ax.set_xlabel('max_reject_rate (cap)')
ax.set_ylabel('FID-10K (lower is better)')
ax.set_title('Block B — metric comparison at 16 steps, τ=0.5')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Block C — step-count trend (if run)

If we also sampled 8-step and 12-step runs, add them to the step-count plot.

In [ ]:
v_by_steps = all_results[all_results['mode'] == 'vanilla'].groupby('steps')['fid'].first()
r_by_steps = all_results[
    (all_results['mode'] == 'rejection') &
    (all_results['metric'] == 'max_prob') &
    (all_results['tau'] == 0.5) &
    (all_results['cap'] == 0.2)
].groupby('steps')['fid'].first()

print("Step-count sweep (max_prob, τ=0.5, cap=0.2):\n")
print(f"{'steps':>6} {'vanilla':>10} {'rejection':>12} {'delta':>10}")
print('-' * 42)
common = sorted(set(v_by_steps.index) & set(r_by_steps.index))
for s in common:
    v = v_by_steps[s]; r = r_by_steps[s]
    marker = "  <--" if r < v else ""
    print(f"{int(s):>6} {v:>10.3f} {r:>12.3f} {r-v:>+10.3f}{marker}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
if len(common):
    xs = common
    ax.plot(xs, [v_by_steps[s] for s in xs], marker='s', linewidth=2,
            markersize=10, label='vanilla', color='tab:gray')
    ax.plot(xs, [r_by_steps[s] for s in xs], marker='o', linewidth=2,
            markersize=10, label='rejection (max_prob, τ=0.5, cap=0.2)', color='tab:blue')
    ax.set_xlabel('decoding steps')
    ax.set_ylabel('FID-10K (lower is better)')
    ax.set_title('Step-count sensitivity (all runs combined)')
    ax.set_xticks(xs)
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 6. Rank the 16-step rejection configs

In [ ]:
r16 = all_results[
    (all_results['mode'] == 'rejection') &
    (all_results['steps'] == 16)
].copy().sort_values('fid')

if vanilla_16 is not None:
    r16['delta_vs_vanilla'] = r16['fid'] - vanilla_16
    r16['pct_vs_vanilla'] = (r16['fid'] - vanilla_16) / vanilla_16 * 100

cols = ['metric', 'tau', 'cap', 'fid', 'is_score', 'precision', 'recall']
if 'delta_vs_vanilla' in r16.columns:
    cols += ['delta_vs_vanilla', 'pct_vs_vanilla']

print("=== 16-step rejection configs, ranked by FID ===\n")
r16_display = r16[cols].reset_index(drop=True).round(3)
r16_display.index = r16_display.index + 1
r16_display

In [ ]:
# Winner summary
best = r16.iloc[0]
print("=" * 60)
print("16-STEP WINNER")
print("=" * 60)
print(f"  metric: {best['metric']}")
print(f"  tau:    {best['tau']}")
print(f"  cap:    {best['cap']}")
print(f"  FID:    {best['fid']:.3f}")
if vanilla_16 is not None:
    print(f"  Vanilla (16 steps) FID: {vanilla_16:.3f}")
    print(f"  Δ: {best['fid'] - vanilla_16:+.3f}")
    print(f"  %: {(best['fid'] - vanilla_16) / vanilla_16 * 100:+.2f}%")

# Robustness check: how many configs beat vanilla?
beat_vanilla = (r16['fid'] < vanilla_16).sum() if vanilla_16 is not None else None
print(f"\nRobustness: {beat_vanilla} / {len(r16)} configs beat vanilla at 16 steps")

## 7. Save to Drive

In [ ]:
DRIVE_RESULTS = f"{DRIVE_ARPG}/results/pilot-20260421/phase2-16step"
!mkdir -p {DRIVE_RESULTS}/logs

# Copy the CSV, logs, and heatmaps
!cp {SWEEP_CSV} {DRIVE_RESULTS}/16step-results.csv
!cp {SWEEP_DIR}/logs/*.json {DRIVE_RESULTS}/logs/ 2>/dev/null || echo "(no logs)"
!cp {SWEEP_DIR}/logs/*_heatmap.png {DRIVE_RESULTS}/logs/ 2>/dev/null || echo "(no heatmaps)"

# Save the merged combined CSV (overwrites previous)
all_results.to_csv(f"{SWEEP_DIR}/combined-results-v2.csv", index=False)
!cp {SWEEP_DIR}/combined-results-v2.csv {DRIVE_RESULTS}/combined-results-v2.csv

print(f"\nResults backed up to: {DRIVE_RESULTS}")
!ls -lh {DRIVE_RESULTS}

## 8. Next step: FID-50K validation

Take the winning 16-step config from the table above and re-run at 50,000 samples to replace the noisy FID-10K estimate with a paper-grade FID-50K number. That becomes the headline number for the progress report.

If the 16-step winner is substantially better than vanilla at 32 steps, we also have a compute-efficiency claim for the report ("16-step rejection approaches 32-step vanilla quality at ~half the compute").